# NHL Data Pipeline — Teams & Standings

In [ ]:
import requests
import sqlite3
import pandas as pd

DB_PATH = "../data/nhl.db"
SCHEMA_PATH = "../sql/create_tables.sql"
STANDINGS_URL = "https://api-web.nhle.com/v1/standings/now"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

with open(SCHEMA_PATH) as f:
    conn.executescript(f.read())
conn.commit()

In [ ]:
resp = requests.get(STANDINGS_URL, timeout=15)
resp.raise_for_status()
standings_raw = resp.json()["standings"]

In [ ]:
teams = pd.DataFrame([{
    "team_abbrev": t["teamAbbrev"]["default"],
    "team_name": t["teamName"]["default"],
    "conference_name": t.get("conferenceName"),
    "division_name": t.get("divisionName"),
    "logo_url": t.get("teamLogo"),
} for t in standings_raw]).drop_duplicates(subset="team_abbrev")

for row in teams.itertuples(index=False):
    cur.execute(
        "INSERT OR IGNORE INTO teams (team_abbrev, team_name, conference_name, division_name, logo_url) "
        "VALUES (?, ?, ?, ?, ?)",
        row,
    )
conn.commit()

team_id_lookup = dict(cur.execute("SELECT team_abbrev, team_id FROM teams").fetchall())

In [ ]:
# roadWins from the API maps to away_wins in the schema
standings = pd.DataFrame([{
    "team_id": team_id_lookup[t["teamAbbrev"]["default"]],
    "season": str(t.get("seasonId")),
    "games_played": t.get("gamesPlayed"),
    "wins": t.get("wins"),
    "losses": t.get("losses"),
    "ot_losses": t.get("otLosses"),
    "points": t.get("points"),
    "goals_for": t.get("goalFor"),
    "goals_against": t.get("goalAgainst"),
    "home_wins": t.get("homeWins"),
    "away_wins": t.get("roadWins"),
    "streak_type": t.get("streakCode"),
    "streak_count": t.get("streakCount"),
} for t in standings_raw])

for row in standings.itertuples(index=False):
    cur.execute(
        "INSERT INTO standings (team_id, season, games_played, wins, losses, ot_losses, points, "
        "goals_for, goals_against, home_wins, away_wins, streak_type, streak_count) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        row,
    )
conn.commit()
conn.close()